<a href="https://colab.research.google.com/github/caiorcastro/Figma-CSV-Import/blob/master/Rob%C3%B4_de_SEO_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-cloud-aiplatform google-auth-oauthlib google-auth-httplib2 google-api-python-client

import pandas as pd
from google.colab import auth, drive
from oauth2client.client import GoogleCredentials
from googleapiclient.discovery import build
import time
import json
import random
import google.auth
import io
import googleapiclient.http
import vertexai
from vertexai.generative_models import GenerativeModel
import re

# Autenticar no Google
auth.authenticate_user()

# Configurar serviço do Google Sheets e Drive
creds, _ = google.auth.default()
sheets_service = build('sheets', 'v4', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

# ID da sua planilha
SPREADSHEET_ID = '1gnOPcYzS8cL9xHwUtCbl3n390qXCpHjf8MQY10C8Y4k'
RANGE_NAME = 'A2:D10001'  # Ler até 10000 linhas de dados + cabeçalho
# Nova variável para a coluna H
HTML_DESCRIPTION_RANGE = 'H2:H10001'

# Configurações do Google Cloud
PROJECT_ID = "seo-gemini-1206"
LOCATION = "us-central1"
MODEL_NAME = "gemini-1.0-pro-001"  # Você pode trocar para "gemini-1.5-pro" se desejar

# Montar o Google Drive
drive.mount('/content/drive')

# Criar a pasta "SEO Gemini" no Google Drive
drive_folder_name = "SEO Gemini"
drive_root = "/content/drive/My Drive"

# Criar um objeto de serviço para a API do Google Drive
drive_service = build('drive', 'v3', credentials=creds)

# Procurar a pasta
folders = drive_service.files().list(
    q=f"name='{drive_folder_name}' and mimeType='application/vnd.google-apps.folder' and 'root' in parents",
    spaces='drive',
    fields='files(id, name)'
).execute()

# Criar a pasta se não existir
if not folders['files']:
    file_metadata = {
        'name': drive_folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': ['root']
    }
    folder = drive_service.files().create(body=file_metadata, fields='id').execute()
    seo_folder_id = folder.get('id')
    print(f'Pasta "{drive_folder_name}" criada com ID: {seo_folder_id}')
else:
    seo_folder_id = folders['files'][0]['id']
    print(f'Pasta "{drive_folder_name}" encontrada com ID: {seo_folder_id}')

# Função para ler a planilha
def read_sheet():
    result = sheets_service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=RANGE_NAME
    ).execute()
    values = result.get('values', [])
    return pd.DataFrame(values, columns=['vendor_name', 'game_id', 'game_code', 'game_name'])

# Função para limpar a resposta JSON
def clean_json_response(text):
    """Remove control characters and other non-printable characters from a string."""
    # Remove control characters
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', '', text)

    # Remove non-printable characters
    text = ''.join(c for c in text if c.isprintable())

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

# Função para tentar corrigir o JSON
def fix_json(text):
    """Tenta corrigir erros simples de formatação JSON, como vírgulas ausentes."""
    text = re.sub(r"([}\]]),]", r"\1,\n", text)  # Adiciona vírgula após chaves ou colchetes, se necessário
    return text

# Função para gerar conteúdo com IA
def generate_game_content(game_name, vendor_name):
    meta_titles = [
        f"{game_name} | Jogue Agora na EstrelaBet",
        f"Conheça {game_name} - Experiência Única | EstrelaBet",
        f"{game_name} da {vendor_name} - Jogue na EstrelaBet",
        f"Descubra {game_name} | EstrelaBet Casino",
        f"{game_name} - Aventura Exclusiva | EstrelaBet"
    ]

    print(f"Gerando conteúdo para {game_name}...")

    prompt = f"""
   Crie um conteúdo otimizado para SEO sobre o jogo {game_name} da {vendor_name} para o site EstrelaBet.

    DIRETRIZES DE TOM DE VOZ:
    - Público-alvo: Homens adultos acima de 25 anos
    - Tom direto, profissional e envolvente
    - Linguagem madura e objetiva
    - Foco na experiência e emoção do jogo
    - Cada texto deve ser único e específico
    - Use termos populares, mas evite gírias, linguajar de baixo escalão ou fora de contexto.

    PERSONALIDADE DA MARCA ESTRELABET:
    - Inovadora
    - Confiável
    - Divertida
    - Emocionante
    - Focada em proporcionar a melhor experiência de apostas online

    EVITAR COMPLETAMENTE:
    - RTP, volatilidade ou retorno do jogo
    - Valores de apostas (mínimas ou máximas)
    - Multiplicadores específicos
    - Número de rolos, colunas ou linhas (qualquer informação que pode variar de casa pra casa ou atualização nova do jogo). Não use valores específicos.
    - Linhas de pagamento
    - Especificações técnicas do jogo
    - Qualquer detalhe que possa variar entre casas de apostas

    FOCO DO CONTEÚDO:
    - Tema e ambiente do jogo: Descreva em detalhes a ambientação, o cenário e a história por trás do jogo {game_name}. Mergulhe nos detalhes visuais, nos personagens (se houver) e na atmosfera geral. Crie uma imagem vívida para o leitor.
    - Experiência de jogo: Detalhe a experiência de jogar {game_name}. O que o jogador sente? Quais são os desafios e as recompensas? Descreva a interatividade e o dinamismo do jogo. Use exemplos hipotéticos para ilustrar a jogabilidade.
    - Sensações e emoções: Foque nas emoções que {game_name} evoca. É adrenalina pura? É suspense e expectativa? É a satisfação de uma vitória bem merecida? Descreva as sensações de imersão, o desafio e a emoção da vitória.
    - Entretenimento e diversão: Explique por que {game_name} é um jogo divertido e que vai entreter o jogador por horas. Destaque os aspectos que tornam o jogo cativante e envolvente.
    - Interatividade: Descreva as diferentes maneiras pelas quais o jogador pode interagir com {game_name}. Quais são as mecânicas de jogo que tornam a experiência interativa?
    - Elementos visuais e sonoros: Comente sobre a qualidade gráfica de {game_name}. Os gráficos são realistas e imersivos? A trilha sonora complementa a atmosfera do jogo? Descreva os efeitos visuais e sonoros que contribuem para a experiência geral.
    - Jogabilidade geral: Explique o fluxo do jogo {game_name}. Como as rodadas funcionam? Quais são os objetivos? Descreva a jogabilidade de forma envolvente, sem entrar em detalhes técnicos.
    - Recursos especiais genéricos: Mencione, de forma sutil e sem detalhes técnicos, recursos comuns em jogos de cassino online, como rodadas grátis, símbolos especiais (Wilds, Scatters), fases bônus, multiplicadores, etc., caso se apliquem ao jogo em questão. Explique como esses recursos podem aumentar a emoção e as chances de vitória, mas sem revelar números ou detalhes específicos.

    - Benefícios de Jogar na EstrelaBet: Explique por que a EstrelaBet é a melhor plataforma para jogar {game_name} e outros jogos de cassino. Mencione vantagens como:
        - Segurança e confiabilidade da plataforma.
        - Variedade de jogos disponíveis.
        - Promoções e bônus oferecidos (sem citar valores específicos).
        - Facilidade de uso do site e do aplicativo.
        - Suporte ao cliente de qualidade.
        - Site Reguamentado (promove segurança ao jogador.)

    PALAVRAS-CHAVE A INCLUIR NATURALMENTE (USE VARIAÇÕES SEMÂNTICAS E SINÔNIMOS SEMPRE QUE POSSÍVEL):
    - {game_name}
    - EstrelaBet
    - cassino online
    - jogo de cassino
    - caça-níqueis
    - slots online
    - apostas online
    - jogar online
    - diversão online
    - entretenimento online
    - emoção
    - adrenalina
    - prêmios
    - vitória
    - sorte
    - bônus de cassino
    - rodadas grátis
    - experiência de jogo
    - melhor cassino online
    - apostas esportivas (se aplicável à EstrelaBet)
    - jogo responsável

    METADATA:
    - Meta Title: Um dos títulos a seguir:
        - {game_name} | Jogue Agora na EstrelaBet
        - Conheça {game_name} - Experiência Única | EstrelaBet
        - {game_name} da {vendor_name} - Jogue na EstrelaBet
        - Descubra {game_name} | EstrelaBet Casino
        - {game_name} - Aventura Exclusiva | EstrelaBet
    - Meta Description: Descrição curta e atraente do jogo em até 157 caracteres, incluindo call-to-action.

    Forneça o resultado exatamente neste formato JSON:
    {{
        "meta_description": "[Descrição seguindo as diretrizes acima, com no máximo 157 caracteres]",
        "meta_title": "{random.choice(meta_titles)}",
        "meta_keywords": "[Palavras-chave relacionadas ao jogo {game_name}, à {vendor_name}, cassino online, EstrelaBet e jogos online. Separe as palavras-chave por vírgula.]",
        "html_description": "<div class='game-description'>
            <h1>{game_name}</h1>

            <h2>Sobre o jogo {game_name}</h2>
            <p>[Introdução focada na temática e experiência do jogo, usando a palavra-chave {game_name}. Aqui você pode detalhar o tema, a ambientação e a história do jogo.]</p>

            <h2>O melhor de {game_name}</h2>
            <p>[Descreva os aspectos mais emocionantes e os elementos de entretenimento que tornam este jogo único. Foque nas sensações, na imersão e nos recursos especiais genéricos. Aqui você pode detalhar a experiência de jogo, as sensações e emoções, a interatividade e os elementos visuais e sonoros.]</p>

            <h2>Como Jogar {game_name}</h2>
            <p>[Visão geral da jogabilidade, sem detalhes técnicos. Explique como o jogo funciona de forma genérica, focando na experiência do usuário.]</p>

            <h2>Dicas e truques para {game_name}</h2>
            <p>[Sugestões gerais de aproveitamento do jogo. Como o jogador pode maximizar sua diversão e emoção ao jogar {game_name}?]</p>

            <h2>Por que jogar {game_name} na EstrelaBet?</h2>
            <p>[Destaque os benefícios de jogar na EstrelaBet, como segurança, variedade de jogos, promoções (sem valores), facilidade de uso e suporte ao cliente.]</p>

            <h2>Perguntas Frequentes</h2>
            <div class='faq-section'>
                <h3>Como começar a jogar {game_name}?</h3>
                <p>[Resposta com call-to-action, focada na experiência e em como é fácil começar a jogar na EstrelaBet]</p>

                <h3>Quais as vantagens de jogar {game_name} na EstrelaBet?</h3>
                <p>[Reforce os benefícios de escolher a EstrelaBet como plataforma de jogos online.]</p>

                <h3>{game_name} é um jogo seguro?</h3>
                <p>[Fale sobre a segurança e a confiabilidade da EstrelaBet, mencionando jogo responsável e licenças, se aplicável.]</p>
            </div>
        </div>"
    }}

    IMPORTANTE:
    - Crie um conteúdo entre 600-800 palavras.
    - Foque na experiência e entretenimento.
    - Evite qualquer especificação técnica.
    - Integre as palavras-chave de forma natural, usando variações e sinônimos.
    - Use tags HTML para formatar o texto. Use <p> para delimitar os parágrafos e <h2> ou <h3> para as seções.
    - Mantenha o conteúdo único e envolvente.
    - O JSON de resposta não deve conter caracteres de controle inválidos ou sequências de escape impróprias. Certifique-se de que o JSON seja válido. **Atenção:** O JSON deve ser estritamente válido e seguir todas as regras de formatação JSON. Não inclua nenhum caractere ou sequência de escape inválida. Certifique-se de que todas as strings estejam entre aspas duplas e que as vírgulas estejam nos lugares corretos.
    """

    try:
        vertexai.init(project=PROJECT_ID, location=LOCATION)
        model = GenerativeModel(MODEL_NAME)
        response = model.generate_content(
            prompt,
            generation_config={
                "max_output_tokens": 8000,
                "temperature": 0.8,
            },
        )

        content = response.text

        # Limpeza da resposta (remoção de caracteres de controle)
        content = clean_json_response(content)

        # Tentativa de correção de JSON
        content = fix_json(content)

        content_json = json.loads(content)
        return content_json

    except Exception as e:
        print(f"Erro na geração de conteúdo para {game_name}: {str(e)}")
        return None

# Função para atualizar a planilha
def update_sheet(row_number, meta_desc, meta_title, meta_keywords, html_desc):
    range_name = f'E{row_number}:H{row_number}'
    values = [[meta_desc, meta_title, meta_keywords, html_desc]]

    body = {
        'values': values
    }

    try:
        sheets_service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=range_name,
            valueInputOption='RAW',
            body=body
        ).execute()
        return True
    except Exception as e:
        print(f"Erro ao atualizar planilha: {str(e)}")
        return False

# Função para salvar o HTML no Google Drive
def save_html_to_drive(game_name, html_content):
    file_name = f"{game_name.replace(' ', '_')}.html"
    file_metadata = {
        'name': file_name,
        'parents': [seo_folder_id]
    }
    media = googleapiclient.http.MediaIoBaseUpload(
        io.BytesIO(html_content.encode('utf-8')),
        mimetype='text/html',
        resumable=True
    )
    file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id'
    ).execute()
    print(f'Arquivo "{file_name}" salvo no Google Drive com ID: {file.get("id")}')

# Processo principal
def main():
    print("Iniciando processamento...")
    df = read_sheet()

    # Ler a coluna H para verificar quais jogos já foram processados
    processed_games = sheets_service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=HTML_DESCRIPTION_RANGE
    ).execute().get('values', [])

    # Achatar a lista (caso tenha mais de uma coluna)
    processed_games = [item[0] if item else '' for item in processed_games]

    for index, row in df.iterrows():
        # Verificar se o jogo já foi processado
        if index < len(processed_games) and processed_games[index] != '':
            print(f"Jogo {row['game_name']} já foi processado. Pulando.")
            continue

        try:
            print(f"\nProcessando jogo: {row['game_name']}")

            # Gerar conteúdo
            content = generate_game_content(row['game_name'], row['vendor_name'])

            if content:
                # Atualizar planilha
                success = update_sheet(
                    index + 2,  # +2 porque o index do pandas começa em 0 e a planilha tem cabeçalho na linha 1
                    content['meta_description'],
                    content['meta_title'],
                    content['meta_keywords'],
                    content['html_description']
                )

                if success:
                    print(f"✓ Atualizado com sucesso: {row['game_name']}")

                    # Salvar HTML no Google Drive
                    save_html_to_drive(row['game_name'], content['html_description'])
                else:
                    print(f"✗ Falha ao atualizar: {row['game_name']}")

                time.sleep(10)  # Manter o tempo de espera

        except Exception as e:
            print(f"Erro ao processar {row['game_name']}: {str(e)}")
            continue

if __name__ == "__main__":
    main()

A saída de streaming foi truncada nas últimas 5000 linhas.
Jogo 9 Jokers Stick and Spin já foi processado. Pulando.
Jogo Coin Miner 2 já foi processado. Pulando.
Jogo Luke E. Chance and the Book of Luck já foi processado. Pulando.
Jogo Baccarat já foi processado. Pulando.
Jogo Paddy's Payouts já foi processado. Pulando.
Jogo Wild Woof já foi processado. Pulando.
Jogo Plinkgoal já foi processado. Pulando.
Jogo Prospector's Plinko já foi processado. Pulando.
Jogo Shootout Champion já foi processado. Pulando.
Jogo Super Hot Stacks já foi processado. Pulando.
Jogo Ramen Puzzle já foi processado. Pulando.
Jogo Destino Das Almas já foi processado. Pulando.
Jogo Mighty Mammoth já foi processado. Pulando.
Jogo Golden Dragon Wealth já foi processado. Pulando.
Jogo Piggy Smash já foi processado. Pulando.
Jogo Vegas Payouts Rapid Hold & Win já foi processado. Pulando.
Jogo RAMPAGE! já foi processado. Pulando.
Jogo Bonbon Bonanza já foi processado. Pulando.
Jogo Big Bang Mines já foi processado. P